# Notebook 02: Leak-Free Preprocessing & Sliding Window Segmentation

**Objective**: Transform raw continuous OPPORTUNITY recordings into normalized sliding-window tensors without data leakage.

### Research Integrity Safeguards:
1. **Zero Data Leakage**: Scaler statistics (mean, std) are computed strictly across training files/subjects.
2. **Continuous Windowing**: Windows are segmented within individual continuous runs, preventing boundary contamination.
3. **Mode Labeling**: The window label is determined by the most frequent activity state within the window.

## 0. Environment Setup & Repository Bootstrap

In [ ]:
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_DIR = Path("/content/Opportunity_HAR")
    if not REPO_DIR.exists():
        print("Cloning Opportunity_HAR from GitHub...")
        !git clone https://github.com/DhruvikaRajput/Opportunity_HAR.git {REPO_DIR}
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
else:
    current = Path.cwd()
    REPO_DIR = current.parent if current.name == "notebooks" else current
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)

import src.api as api
env_info = api.setup_project()
print(f"Project Initialized on: {env_info['device_name']} (CUDA: {env_info['cuda_available']})")

## 1. Locate Dataset & Run Preprocessing Pipeline
Preprocesses on-body sensor channels for the Locomotion track (5 states: Stand, Walk, Sit, Lie, Null).

In [ ]:
ds_status = api.locate_dataset()
print(ds_status["message"])

if ds_status["found"]:
    summary = api.preprocess_data(
        track="Locomotion",
        output_filename="opportunity_locomotion_subset.npz",
        window_size=30,
        stride=15
    )
    print("\nPreprocessing Complete!")
    print(f"Total Windows Generated: {summary['total_windows']:,}")
    print(f"Class Distribution: {summary['class_distribution']}")
else:
    print("\n[ACTION NEEDED] Raw dataset not found. If in Colab, mount Google Drive in Notebook 00 and place .dat files in Drive.")

## 2. Verify Generated Sliding-Window Tensor Shapes

In [ ]:
try:
    subset = api.load_preprocessed_subset("opportunity_locomotion_subset.npz")
    X_train, y_train = subset["X_train"], subset["y_train"]
    X_val, y_val = subset["X_val"], subset["y_val"]
    X_test, y_test = subset["X_test"], subset["y_test"]

    print("Tensor Shapes:")
    print(f"  X_train: {X_train.shape}  |  y_train: {y_train.shape}")
    print(f"  X_val  : {X_val.shape}  |  y_val  : {y_val.shape}")
    print(f"  X_test : {X_test.shape}  |  y_test : {y_test.shape}")
    print(f"  Classes: {subset['class_names']}")

    assert X_train.shape[1] == 133, "Expected 133 on-body sensor channels"
    assert X_train.shape[2] == 30, "Expected 30 timesteps per window (1.0 sec)"
    print("\n[OK] All tensor shapes verified successfully!")
except FileNotFoundError as e:
    print(e)

## 3. Visualize a Sample Windowed Tensor

In [ ]:
try:
    import matplotlib.pyplot as plt
    sample_idx = 42
    sample_window = X_train[sample_idx]  # Shape: (133, 30)
    sample_label = subset['class_names'][y_train[sample_idx]]

    plt.figure(figsize=(10, 4))
    plt.plot(sample_window[0:3].T, label=["Back_Acc_X", "Back_Acc_Y", "Back_Acc_Z"])
    plt.title(f"Sample Normalized Window #{sample_idx} - Activity: {sample_label}")
    plt.xlabel("Timestep in Window (0 to 30)")
    plt.ylabel("Standardized Amplitude")
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Skipping visualization:", e)